# Lab 3 Exercise 3
## Binary Classification

We are given a dataset that consists of biological features of a person and whether or not they are a smoker.

There are *22 descriptive features* and *1 target column* in the dataset as described in the Table below. You are tasked with creating a predictive model to predict the column "SMK_stat_type_cd", which specifies smoker/non-smoker status of the person. You are expected to use classification methods that was shown in the previous exercises.



|      Column      |                                 Description                                 |
|:----------------:|:---------------------------------------------------------------------------:|
| identified_gender              | male, female                                                                |
| age              | round up to 5 years                                                         |
| height           | round up to 5 cm[cm]                                                        |
| weight           | [kg]                                                                        |
| sight_left       | eyesight(left)                                                              |
| sight_right      | eyesight(right)                                                             |
| hear_left        | hearing left, 1(normal), 2(abnormal)                                        |
| hear_right       | hearing right, 1(normal), 2(abnormal)                                       |
| SBP              | Systolic blood pressure[mmHg]                                               |
| DBP              | Diastolic blood pressure[mmHg]                                              |
| BLDS             | BLDS or FSG(fasting blood glucose)[mg/dL]                                   |
| tot_chole        | total cholesterol[mg/dL]                                                    |
| HDL_chole        | HDL cholesterol[mg/dL]                                                      |
| LDL_chole        | LDL cholesterol[mg/dL]                                                      |
| triglyceride     | triglyceride[mg/dL]                                                         |
| hemoglobin       | hemoglobin[g/dL]                                                            |
| urine_protein    | protein in urine, 1(-), 2(+/-), 3(+1), 4(+2), 5(+3), 6(+4)                  |
| serum_creatinine | serum(blood) creatinine[mg/dL]                                              |
| SGOT_AST         | SGOT(Glutamate-oxaloacetate transaminase) AST(Aspartate transaminase)[IU/L] |
| SGOT_ALT         | ALT(Alanine transaminase)[IU/L]                                             |
| gamma_GTP        | y-glutamyl transpeptidase[IU/L]                                             |
| SMK_stat_type_cd **(Target)** | Smoking state, 0(never), 1(active smoker)                      |


### The stages of this problem can be decomposed as follows:
1. Data Preparation
* Ensure data is in correct format (numerical and not string)
* Normalize the data for better convergence (optional)
* Split the data into train/test subsets

2. Model Selection
* Instanciate models and fit on training data
* Evaluate model performance on testing data
* Select model with best performance

3. Submit model
* Your model will be evaluated on data that is kept separate from training/testing data
* The predictions from your model will be uploaded to the course server where it will be evaluated

## 📘 Lab 3 Ex. 3 study guide: an end-to-end classification project

This exercise puts Labs 1–3 together into the standard ML workflow (handout-04 slides 34–38):

1. **Prepare data**: read the CSV, encode categorical columns as numbers, separate features $X$ from target $y$.
2. **Split**: train / test. The hidden `validation.csv` acts as the final, unseen test set for marking.
3. **Fit several models** on the training split only.
4. **Compare** them on held-out data with a suitable metric (here **F1**, because ~38% of people are smokers, so the classes are mildly imbalanced).
5. **Choose** a model, predict on `validation.csv`, save `lab3_ex3_preds.npz`, and push.

**What is graded:** the predictions file. The server computes your score on labels you never see, so the goal is **generalisation**, not training performance.

*(The table lists 22 features, but the CSV also has a `waistline` column, so there are actually 23 input features.)*

In [48]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/4c16-labs/code/dist/lab-03/
!unzip -o data.zip
!echo 'data/*' > .gitignore

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/4c16-labs/code/dist/lab-03
Archive:  data.zip
  inflating: data/data.csv           
  inflating: data/validation.csv     


In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn.preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

In [50]:
# Data Preparation
# The data is supplied as ".csv" format
# There are 23 columns in csv file, with 22 columns being features and 1 column being the target

# The csv file can be read into as a dataframe using the pandas library
DataFrame = pd.read_csv('data/data.csv')

# Shuffle the data
DataFrame = DataFrame.sample(frac=1)

# Visualize the first 5 rows of the imported dataframe
display(DataFrame.head(6))

,identified_gender,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,HDL_chole,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd
15817,Female,75,160,65,80.0,0.5,0.1,1.0,1.0,131.0,...,49.0,144.0,120.0,10.3,1.0,0.5,20.0,15.0,10.0,0.0
28976,Male,25,160,55,77.0,1.5,1.5,1.0,1.0,120.0,...,40.0,103.0,56.0,15.5,1.0,0.7,37.0,74.0,16.0,0.0
45426,Female,75,145,50,80.0,0.8,0.7,1.0,1.0,140.0,...,48.0,117.0,157.0,11.9,1.0,0.8,19.0,18.0,18.0,0.0
10140,Male,55,160,80,100.0,1.5,1.2,1.0,1.0,128.0,...,46.0,139.0,88.0,14.6,1.0,1.1,31.0,26.0,22.0,0.0
12175,Male,25,165,60,72.0,0.4,0.4,1.0,1.0,124.0,...,51.0,60.0,69.0,15.2,1.0,0.9,23.0,27.0,25.0,0.0
10259,Female,50,155,40,68.0,0.9,0.9,1.0,1.0,144.0,...,57.0,130.0,102.0,12.4,2.0,0.7,19.0,13.0,14.0,0.0


In [51]:
# The column "identified_gender" contains non-numeric data. This cannot be used as is and needs to be converted to numerical representations
# We can encode the identified gender to a numerical representation by letting "female" == 0, and "male" == 1.
# This can be done manually by using conditional statements on the dataframe, but luckily for us there is a simpler method
# We can use the built in LabelEncoder function of sklearn to do this

LabelEncoder = sklearn.preprocessing.LabelEncoder()
LabelEncoder.fit(DataFrame['identified_gender'])
DataFrame['identified_gender'] = LabelEncoder.transform(DataFrame['identified_gender'])

# Label encoder sorts the input column of data in alphabetical order, and then assigns a numerical value to each unique entry
# This results in 'female' being mapped to 0, 'male' being mapped to 1
display(DataFrame.head(6))

,identified_gender,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,HDL_chole,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd
15817,0,75,160,65,80.0,0.5,0.1,1.0,1.0,131.0,...,49.0,144.0,120.0,10.3,1.0,0.5,20.0,15.0,10.0,0.0
28976,1,25,160,55,77.0,1.5,1.5,1.0,1.0,120.0,...,40.0,103.0,56.0,15.5,1.0,0.7,37.0,74.0,16.0,0.0
45426,0,75,145,50,80.0,0.8,0.7,1.0,1.0,140.0,...,48.0,117.0,157.0,11.9,1.0,0.8,19.0,18.0,18.0,0.0
10140,1,55,160,80,100.0,1.5,1.2,1.0,1.0,128.0,...,46.0,139.0,88.0,14.6,1.0,1.1,31.0,26.0,22.0,0.0
12175,1,25,165,60,72.0,0.4,0.4,1.0,1.0,124.0,...,51.0,60.0,69.0,15.2,1.0,0.9,23.0,27.0,25.0,0.0
10259,0,50,155,40,68.0,0.9,0.9,1.0,1.0,144.0,...,57.0,130.0,102.0,12.4,2.0,0.7,19.0,13.0,14.0,0.0


In [52]:
# We should now separate the input features from the target feature and store them as different variables
# We do this by slicing what columns of data we want from the dataframe
# Let's first see the columns available to us by printing DataFrame.columns
print(DataFrame.columns)

Index(['identified_gender', 'age', 'height', 'weight', 'waistline',
       'sight_left', 'sight_right', 'hear_left', 'hear_right', 'SBP', 'DBP',
       'BLDS', 'tot_chole', 'HDL_chole', 'LDL_chole', 'triglyceride',
       'hemoglobin', 'urine_protein', 'serum_creatinine', 'SGOT_AST',
       'SGOT_ALT', 'gamma_GTP', 'SMK_stat_type_cd'],
      dtype='object')


In [53]:
# The column titled "SMK_stat_type_cd" is our target, and all remaining variables are our input features.
features_columns = DataFrame.columns[:-1]
target_column = DataFrame.columns[-1:]

DataFrame_X = DataFrame[features_columns]   # Selects only Input features
DataFrame_Y = DataFrame[target_column]      # Selects only Target variable

In [54]:
# Split Data into Train/Test split
# Change the variable "test_frac" to reflect the percentage/fraction of test data in the resulting split
test_frac = 0.15

Train_X, Test_X, Train_Y, Test_Y = train_test_split(DataFrame_X, DataFrame_Y, test_size=test_frac)
print(f"Number of rows for Training:\t{Train_X.shape[0]}\nNumber of rows for Testing:\t{Test_X.shape[0]}")

Number of rows for Training:	42500
Number of rows for Testing:	7500


### 💡 Notes on data preparation
- **Shuffle:** `DataFrame.sample(frac=1)` has no `random_state`, so the split and the scores change slightly on every run. Add `random_state=0` (here and in `train_test_split`) if you want reproducible comparisons.
- **Label encoding:** `LabelEncoder` sorts the categories alphabetically, so Female → 0 and Male → 1. The *same fitted encoder* must be reused on `validation.csv` (the last cell does this). Re-fitting on new data could assign different codes.
  - For categorical features with more than 2 values, prefer **one-hot encoding**. Integer codes would imply a false ordering.
- **Gender is highly informative here:** about 66% of men in the data are smokers, versus about 9% of women (verified).
- **Split:** `test_frac = 0.15` gives 42,500 training and 7,500 test rows. That is plenty of test data for a stable F1 estimate.

In [ ]:
# EDIT THIS CELL
# Construct a dictionary of prediction models to compare
# Uncomment the below dictionary and insert as many prediction models as you like.
# You may have used binary classification models in previous exercises
# You may also have to import these modules/libraries to be able to use them

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PredictionModels = {
    'Logistic Regression': make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=0,
        n_jobs=-1,
    ),
    'Histogram Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=200,
        class_weight='balanced',
        random_state=0,
    ),
}

models = list(PredictionModels.keys())
train_f1_scores = {}
test_f1_scores = {}

print("Fitting models, this may take a while")
for _model in models:
    model = PredictionModels[_model]
    model.fit(Train_X, Train_Y.values.ravel())

    train_predictions = model.predict(Train_X)
    test_predictions = model.predict(Test_X)
    train_f1_scores[_model] = f1_score(Train_Y.values.ravel(), train_predictions)
    test_f1_scores[_model] = f1_score(Test_Y.values.ravel(), test_predictions)

    print(
        f"{_model}: train F1 = {train_f1_scores[_model]:.4f}, "
        f"test F1 = {test_f1_scores[_model]:.4f}"
    )

Fitting models, this may take a while
Logistic Regression: train F1 = 0.7503, test F1 = 0.7468
Random Forest: train F1 = 1.0000, test F1 = 0.7398


### 💡 Concept note: the three candidate models

| Model | Idea | Needs scaling? | Typical behaviour |
|---|---|---|---|
| **Logistic regression** (in a `make_pipeline(StandardScaler(), …)`) | linear boundary, Lab 2 | yes; the pipeline fits the scaler on training data only (no leakage) | stable; train ≈ test, so little overfitting |
| **Random forest** (200 trees) | average of many deep decision trees trained on bootstrap samples (handout-03 slides 6–10) | no | fits the training set almost perfectly (train F1 = 1.0), so the train/test gap reveals overfitting, but averaging still generalises reasonably |
| **Histogram gradient boosting** (200 iterations) | adds shallow trees one after another, each correcting the previous errors (boosting, handout-03 slides 6–10) | no | usually the strongest model on tabular data |

`class_weight='balanced'` up-weights the minority class (smokers), which typically raises recall and F1 (compare Ex. 1 Q2).

### 🔍 Interpreting the results, and how to choose `chosen_model`

**Saved output in this notebook:**
- Logistic regression: train F1 0.750 / test F1 0.747
- Random forest: train F1 **1.000** / test F1 0.740
- **The histogram-gradient-boosting line is missing.** The cell was interrupted before the third model finished. Re-run it.

**Verified locally** (same models, seeded 85/15 split):

| Model | Train F1 | Test F1 | Test accuracy |
|---|---|---|---|
| Logistic Regression | 0.751 | 0.750 | 0.791 |
| Random Forest | 1.000 | 0.754 | 0.793 |
| **Histogram Gradient Boosting** | 0.779 | **0.760** | 0.794 |

**How to read this**
- **Random forest:** a train F1 of 1.0 vs a test F1 of ~0.75 is a textbook overfitting gap. Test performance is what counts.
- **Logistic regression:** almost no gap. It is limited by its linear boundary (it underfits slightly), not by variance.
- **Gradient boosting:** a small gap and the best test F1. This is the usual result on tabular data.
- **Differences of about 0.01 F1 are close to split-to-split noise.** To decide robustly, use **k-fold cross-validation** (`sklearn.model_selection.cross_val_score(model, X, y, cv=5, scoring='f1')`).

**Recommendation:** after re-running, set `chosen_model = 'Histogram Gradient Boosting'` if it again has the highest test F1. Otherwise keep the model with the highest **test** (not train) F1. Then re-run the final cell to regenerate `lab3_ex3_preds.npz`, and push.

In [ ]:
# Select the model with best f1 score and write down the
# corresponding 'key' value from the 'PredictionModels' variable
# Eg if Logistic Regression is chosen then: chosen_model = 'Logistic Regression'
chosen_model = 'Logistic Regression'

In [ ]:
# DO NOT EDIT.
# Generate predictions using "chosen_model" and save to file

backend_features = pd.read_csv('data/validation.csv')
backend_features['identified_gender'] = LabelEncoder.transform(backend_features['identified_gender'])
backend_preds = PredictionModels[chosen_model].predict(backend_features)
np.savez_compressed('lab3_ex3_preds', lab3_model=backend_preds)

# Remember to push your changes to the git server for marking!

### ✅ Lab 3 Ex. 3 key takeaways
1. A real project = preparation → split → fit several models → compare on held-out data → predict on truly unseen data.
2. Apply **identical** preprocessing (encoders, scalers) to training, test and deployment data. Pipelines make this automatic and prevent leakage.
3. Judge models by **test** metrics. A perfect training score (random forest) signals overfitting, not quality.
4. Pick the metric to suit the problem (F1 for imbalanced classes), and remember that small differences may be noise.